In [0]:
from pyspark.sql import functions as F

raw_df = spark.read.parquet("s3://ledgr-raw-data-2026/processed/")

print(f"Rows read from S3: {raw_df.count()}")
print(f"Columns: {raw_df.columns}")

In [0]:
from delta.tables import DeltaTable

bronze_table = "ledgr.bronze.calls_raw"

if spark.catalog.tableExists(bronze_table):
    # Table exists: MERGE to dedupe on attempt_id, avoiding double-counting on reruns
    delta_table = DeltaTable.forName(spark,bronze_table)
    (delta_table.alias("target")
    .merge(raw_df.alias("source"), "target.attempt_id = source.attempt_id")
    .whenNotMatchedInsertAll()
     .execute())
    print("Merged into existing Bronze table (idempotent, no duplicates)")
else:
     # First run: create the table
     raw_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)
     print(f"Created new Bronze table: {bronze_table}")
# Verify
result_count = spark.table(bronze_table).count()
print(f"Bronze table row count: {result_count}")

In [0]:
# Idempotency test: rerun the same read+merge logic and confirm no duplication
raw_df_rerun = spark.read.parquet("s3://ledgr-raw-data-2026/processed/")

from delta.tables import DeltaTable
delta_table = DeltaTable.forName(spark,bronze_table)
(delta_table.alias("target")
.merge(raw_df_rerun.alias("source"), "target.attempt_id = source.attempt_id")
.whenNotMatchedInsertAll()
.execute())

recount = spark.table("ledgr.bronze.calls_raw").count()
print(f"Row count after rerun: {recount}")
print(f"Expected (no duplication): 265091")
print(f"MATCH: {recount == 265091}")

In [0]:
# Drop the incorrectly-scoped Bronze table from earlier
spark.sql("DROP TABLE IF EXISTS ledgr.bronze.calls_raw")
print("Dropped incorrect Bronze table")

# Read the TRUE raw dataset — untouched, nested spans column, one row per session
true_raw_df = spark.read.parquet("s3://ledgr-raw-data-2026/raw/")
print(f"Raw rows read: {true_raw_df.count()}")
print(f"Columns: {true_raw_df.columns}")
true_raw_df.printSchema()

In [0]:
bronze_table = "ledgr.bronze.sessions_raw"

true_raw_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)
print(f"Created Bronze table: {bronze_table}")

result_count = spark.table(bronze_table).count()
print(f"Bronze table row count: {result_count}")

In [0]:
%pip install pytest

In [0]:
import subprocess
result = subprocess.run(["pytest", "/Workspace/Users/sreelakshmitd97@gmail.com/ledgr/databricks/tests/", "-v"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [0]:
# Add Delta CHECK constraints for data quality enforcement
spark.sql("""
    ALTER TABLE ledgr.bronze.sessions_raw 
    ADD CONSTRAINT session_id_not_null CHECK (session_id IS NOT NULL)
""")

spark.sql("""
    ALTER TABLE ledgr.bronze.sessions_raw 
    ADD CONSTRAINT run_id_not_null CHECK (run_id IS NOT NULL)
""")

print("Constraints added successfully")

# Verify they're active
constraints = spark.sql("DESCRIBE DETAIL ledgr.bronze.sessions_raw").select("properties").collect()
print(constraints)

In [0]:
import subprocess
import os

env = os.environ.copy()
env["PYTHONDONTWRITEBYTECODE"] = "1"
env["PYTHONPATH"] = "/Workspace/Users/sreelakshmitd97@gmail.com/ledgr:" + env.get("PYTHONPATH", "")

result = subprocess.run(
    ["pytest", "-p", "no:cacheprovider", "--assert=plain",
     "/Workspace/Users/sreelakshmitd97@gmail.com/ledgr/ledgr_databricks/tests/", "-v"],
    capture_output=True, text=True, env=env
)
print(result.stdout)
print(result.stderr)

In [0]:
# Add Delta CHECK constraints for data quality enforcement
spark.sql("""
    ALTER TABLE ledgr.bronze.sessions_raw 
    ADD CONSTRAINT session_id_not_null CHECK (session_id IS NOT NULL)
""")

spark.sql("""
    ALTER TABLE ledgr.bronze.sessions_raw 
    ADD CONSTRAINT run_id_not_null CHECK (run_id IS NOT NULL)
""")

print("Constraints added successfully")

In [0]:
# Check what constraints are already active on the table
result = spark.sql("SHOW TBLPROPERTIES ledgr.bronze.sessions_raw").collect()
for row in result:
    if "constraint" in row.key.lower():
        print(row.key, "=", row.value)

In [0]:
from pyspark.sql import Row

# Get the real Bronze table's schema, so our test row matches exactly
bronze_schema = spark.table("ledgr.bronze.sessions_raw").schema

bad_row_df = spark.createDataFrame([
    Row(schema_version="1.0", config_path=None, run_id="test_run", session_id=None,
        harness="claude_code", benchmark="test", benchmark_subset=None,
        models=["test"], score=0.0, success=True, status="ok", steps=1,
        action_count=1, agent_cost=0.0, benchmark_cost=0.0, execution_time=0.0,
        total_tokens=0, max_tokens=0, spans=[], collected_at="2026-01-01")
], schema=bronze_schema)

try:
    bad_row_df.write.format("delta").mode("append").saveAsTable("ledgr.bronze.sessions_raw")
    print("PROBLEM: bad row was inserted, constraint did not actually block it!")
except Exception as e:
    print(f"CORRECTLY REJECTED by constraint: {type(e).__name__}")
    print(str(e)[:300])

print(f"Row count after failed insert attempt: {spark.table('ledgr.bronze.sessions_raw').count()}")

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW ledgr.gold.sessions_analyst_view AS
SELECT
    session_id,
    run_id,
    harness,
    benchmark,
    success,
    agent_cost,
    execution_time
FROM ledgr.bronze.sessions_raw
""")
print("Analyst view created, excludes config_path, benchmark_subset, and raw spans data")

In [0]:
# Add third Bronze constraint: harness must be one of the known 5 values
spark.sql("""
    ALTER TABLE ledgr.bronze.sessions_raw 
    ADD CONSTRAINT harness_known_value 
    CHECK (harness IN ('claude_code', 'openai_solo', 'smolagents_code', 
                        'tool_calling', 'tool_calling_with_shortlisting'))
""")
print("harness_known_value constraint added")

# Verify with a real rejection test
from pyspark.sql import Row
bronze_schema = spark.table("ledgr.bronze.sessions_raw").schema
bad_harness_row = spark.createDataFrame([
    Row(schema_version="1.0", config_path=None, run_id="test_run", session_id="test_session",
        harness="totally_unknown_harness", benchmark="test", benchmark_subset=None,
        models=["test"], score=0.0, success=True, status="ok", steps=1,
        action_count=1, agent_cost=0.0, benchmark_cost=0.0, execution_time=0.0,
        total_tokens=0, max_tokens=0, spans=[], collected_at="2026-01-01")
], schema=bronze_schema)

try:
    bad_harness_row.write.format("delta").mode("append").saveAsTable("ledgr.bronze.sessions_raw")
    print("PROBLEM: bad harness was inserted")
except Exception as e:
    print(f"CORRECTLY REJECTED: {type(e).__name__}")

print(f"Row count: {spark.table('ledgr.bronze.sessions_raw').count()}")

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW ledgr.gold.sessions_analyst_view AS
SELECT
    session_id,
    run_id,
    harness,
    benchmark,
    success,
    agent_cost,
    execution_time
FROM ledgr.bronze.sessions_raw
""")
print("Analyst view created")

view_cols = spark.table("ledgr.gold.sessions_analyst_view").columns
base_cols = spark.table("ledgr.bronze.sessions_raw").columns
print(f"Base table columns ({len(base_cols)}): {base_cols}")
print(f"Analyst view columns ({len(view_cols)}): {view_cols}")
print(f"Columns hidden from analyst view: {set(base_cols) - set(view_cols)}")

In [0]:
# Grant the restricted group access to ONLY the narrow analyst view
spark.sql("GRANT SELECT ON VIEW ledgr.gold.sessions_analyst_view TO `ledgr_analysts`")
print("Granted SELECT on sessions_analyst_view to ledgr_analysts group")

# Confirm the full Bronze table does NOT have this grant
bronze_grants = spark.sql("SHOW GRANTS ON TABLE ledgr.bronze.sessions_raw").collect()
print("\nGrants on Bronze table (should NOT include ledgr_analysts):")
for row in bronze_grants:
    print(row)

# Confirm the view DOES have the grant
view_grants = spark.sql("SHOW GRANTS ON VIEW ledgr.gold.sessions_analyst_view").collect()
print("\nGrants on the analyst view (should include ledgr_analysts):")
for row in view_grants:
    print(row)